<a href="https://colab.research.google.com/github/manjunath1005/celebal-excellence-internship/blob/main/week7/week7_Manjunath.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Week 7 Assignment RAG System — Custom Document Q&A

This notebook builds a Retrieval-Augmented Generation pipeline over a custom document (my resume): load it, chunk it, embed the chunks, store them in a vector index, retrieve the relevant chunks for a question, and generate a grounded answer with an LLM. A hybrid (keyword + vector) retrieval step is included as the optimization experiment.

**Source document:** `Manjunath_SDE_Resume.pdf`

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers rank_bm25 pdfplumber

In [2]:
!pip install Pillow==12.2.0
import os
import re
import textwrap
import numpy as np
import pandas as pd
import faiss
import pdfplumber
from sentence_transformers import SentenceTransformer    # Restart the session after getting error
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 120)

## 1. Document ingestion

`load_document()` reads a `.txt` or `.pdf` file and returns the raw text.

In [3]:
def load_document(path: str) -> str:
    """Load raw text from a .txt or .pdf file."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".txt":
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    elif ext == ".pdf":
        with pdfplumber.open(path) as pdf:
            return "\n".join(page.extract_text() or "" for page in pdf.pages)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

In [4]:
SOURCE_FILE = "Manjunath_SDE_Resume.pdf"
raw_text = load_document(SOURCE_FILE)
print(f"Loaded {len(raw_text)} characters from {SOURCE_FILE}")

Loaded 2792 characters from Manjunath_SDE_Resume.pdf


## 2. Chunking

Two chunkers, used in order of preference:

1. **`chunk_by_sections()`** — the primary strategy for a structured document like a resume. It splits on the known section headings (Objective, Education, Projects, Technical Skills, Certifications, Achievements) so each chunk is one complete, topically coherent section.
2. **`chunk_text()`** — a general-purpose fallback for documents without recognizable headings. It's a sliding-window chunker with overlap snapped to a word boundary, so a chunk never starts mid-word.

In [5]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """General-purpose line-aware chunking with a sliding-window overlap."""
    units = [u.strip() for u in re.split(r"\n+", text) if u.strip()]

    def word_safe_tail(s: str, n: int) -> str:
        tail = s[-n:]
        space_idx = tail.find(" ")
        return tail[space_idx + 1:] if space_idx != -1 else tail

    chunks, current = [], ""
    for unit in units:
        if len(unit) > chunk_size:
            if current:
                chunks.append(current)
                current = word_safe_tail(current, overlap) if overlap else ""
            start = 0
            while start < len(unit):
                chunks.append(unit[start:start + chunk_size])
                start += max(chunk_size - overlap, 1)
            current = ""
            continue

        if len(current) + len(unit) + 1 <= chunk_size:
            current = f"{current}\n{unit}".strip()
        else:
            chunks.append(current)
            tail = word_safe_tail(current, overlap) if overlap else ""
            current = f"{tail}\n{unit}".strip()
    if current:
        chunks.append(current)
    return chunks


SECTION_HEADINGS = ["Objective", "Education", "Projects", "Technical Skills",
                     "Certifications", "Achievements"]

def chunk_by_sections(text: str, headings: list[str]) -> list[str]:
    """Split on known section headings so each chunk is one complete section.
    Falls back to chunk_text() if none of the headings are found."""
    pattern = "(" + "|".join(re.escape(h) for h in headings) + ")"
    parts = re.split(pattern, text)

    if len(parts) <= 1:
        return chunk_text(text)

    sections = []
    if parts[0].strip():
        sections.append(parts[0].strip())  # header block: name/contact info
    i = 1
    while i < len(parts) - 1:
        heading, body = parts[i], parts[i + 1].strip()
        sections.append(f"{heading}\n{body}")
        i += 2
    return sections


chunks = chunk_by_sections(raw_text, SECTION_HEADINGS)

print(f"Document split into {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"\n--- chunk {i} ({len(c)} chars) ---")
    print(textwrap.shorten(c.replace(chr(10), " "), width=140))

Document split into 7 chunks

--- chunk 0 (146 chars) ---
Manjunath Chikka Phone: +91 8328099522 Email: 10dch.manjunath@gmail.com LinkedIn: linkedin.com/in/manjunathchikka GitHub: [...]

--- chunk 1 (361 chars) ---
Objective Computer Science and Engineering (Data Science) undergraduate with strong foundations in software development, data [...]

--- chunk 2 (227 chars) ---
Education CVR College of Engineering, Hyderabad 2023 – 2027 B.Tech in Computer Science and Engineering (Data Science) CGPA: 8.35/10 [...]

--- chunk 3 (1279 chars) ---
Projects Startup Idea Validation Platform | FastAPI, React, Google Gemini API, Docker • Developed a full-stack platform that analyzes [...]

--- chunk 4 (311 chars) ---
Technical Skills Programming Languages: Python, Java, C, R, MATLAB (Basic) Web & Backend Technologies: Flask, FastAPI, Node.js, React, [...]

--- chunk 5 (243 chars) ---
Certifications AWS Academy – Cloud Foundations Alteryx – Data Analytics Process Automation Virtual Internship AICTE –

## 3. Embedding the chunks

Using `all-MiniLM-L6-v2` from `sentence-transformers` — a small, fast model that maps each chunk to a 384-dimensional vector.

In [6]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embed_model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)
EMBED_DIM = chunk_embeddings.shape[1]
print(f"Embedding matrix shape: {chunk_embeddings.shape} (dim = {EMBED_DIM})")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding matrix shape: (7, 384) (dim = 384)


## 4. Vector Store

The text embeddings are stored in a FAISS vector database so that similar document chunks can be retrieved efficiently. I used `IndexFlatIP`, which works well with normalized embeddings and provides accurate similarity search for this small-scale RAG system.

In [7]:
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(chunk_embeddings)
print(f"FAISS index holds {index.ntotal} vectors of dimension {index.d}")

FAISS index holds 7 vectors of dimension 384


## 5. Query embedding

Incoming questions go through the same embedding model and normalization as the stored chunks, so both live in the same vector space.

In [8]:
def embed_query(question: str) -> np.ndarray:
    return embed_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)

## 6. Retrieval (vector search)

`retrieve()` embeds the query, searches the FAISS index for the top-k nearest chunks, and returns them with their similarity scores.

In [9]:
def retrieve(question: str, k: int = 3) -> list[tuple[str, float]]:
    q_vec = embed_query(question)
    scores, idxs = index.search(q_vec, k)
    return [(chunks[i], float(scores[0][j])) for j, i in enumerate(idxs[0])]


for chunk, score in retrieve("What is the CGPA mentioned in the resume?", k=2):
    print(f"[{score:.3f}] {textwrap.shorten(chunk.replace(chr(10), ' '), 120)}")

[0.534] Education CVR College of Engineering, Hyderabad 2023 – 2027 B.Tech in Computer Science and Engineering (Data [...]
[0.165] Certifications AWS Academy – Cloud Foundations Alteryx – Data Analytics Process Automation Virtual Internship [...]


## 7. Prompt Construction & Answer Generation

The retrieved context is combined with the user's question and sent to the `google/flan-t5-base` model. The prompt encourages the model to answer only from the document and avoid making assumptions.

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

MAX_CONTEXT_CHARS = 900

def build_prompt(question: str, retrieved_chunks: list[str]) -> str:
    per_chunk_cap = max(MAX_CONTEXT_CHARS // max(len(retrieved_chunks), 1), 150)
    context = "\n\n".join(c[:per_chunk_cap] for c in retrieved_chunks)
    return (
        "You are a question-answering assistant.\n"
        "Answer only using the provided context.\n"
        "If the answer is not found in the context, reply: "
        "\"The information is not available in the document.\"\n"
        "Do not guess or use outside knowledge. Answer briefly and directly - "
        "do not copy the entire context.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )

def generate_answer(question: str, retrieved_chunks: list[str]) -> str:
    prompt = build_prompt(question, retrieved_chunks)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=160,
        repetition_penalty=1.15,
    )
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    for heading in SECTION_HEADINGS:
        idx = answer.find(heading)
        if idx > 0:
            answer = answer[:idx].strip()
    return answer

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 8. End-to-End Pipeline (Vector Search)

This pipeline uses vector similarity search to retrieve relevant document chunks and generate answers based on the retrieved context.

In [11]:
def rag_answer(question: str, k: int = 5, verbose: bool = True) -> str:
    retrieved = retrieve(question, k=k)
    retrieved_chunks = [c for c, _ in retrieved]
    answer = generate_answer(question, retrieved_chunks)
    if verbose:
        print(f"Q: {question}")
        for c, s in retrieved:
            print(f"  retrieved [{s:.3f}]: {textwrap.shorten(c.replace(chr(10), ' '), 100)}")
        print(f"A: {answer}\n")
    return answer

rag_answer("What is the CGPA mentioned in the resume?")

Q: What is the CGPA mentioned in the resume?
  retrieved [0.534]: Education CVR College of Engineering, Hyderabad 2023 – 2027 B.Tech in Computer Science and [...]
  retrieved [0.165]: Certifications AWS Academy – Cloud Foundations Alteryx – Data Analytics Process Automation [...]
  retrieved [0.148]: Objective Computer Science and Engineering (Data Science) undergraduate with strong [...]
  retrieved [0.140]: Technical Skills Programming Languages: Python, Java, C, R, MATLAB (Basic) Web & Backend [...]
  retrieved [0.102]: Achievements Solved 150+ coding problems on LeetCode focusing on data structures and [...]
A: 8.35/10



'8.35/10'

## 9. Optimization – Hybrid Search (Keyword + Vector)

To improve retrieval, I combined FAISS vector search with BM25 keyword search. This makes it easier to find the most relevant document chunks for different types of questions.

In [12]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_chunks = [tokenize(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def hybrid_retrieve(question: str, k: int = 3, alpha: float = 0.5) -> list[tuple[str, float]]:
    """alpha weights vector similarity vs. BM25 keyword score (0=keyword only, 1=vector only)."""
    q_vec = embed_query(question)
    vec_scores, vec_idxs = index.search(q_vec, len(chunks))
    vec_score_by_idx = {int(i): float(vec_scores[0][j]) for j, i in enumerate(vec_idxs[0])}

    bm25_scores = bm25.get_scores(tokenize(question))
    bm25_norm = (bm25_scores - bm25_scores.min()) / (np.ptp(bm25_scores) + 1e-9)

    combined = [
        alpha * vec_score_by_idx.get(i, 0.0) + (1 - alpha) * bm25_norm[i]
        for i in range(len(chunks))
    ]
    top_idxs = np.argsort(combined)[::-1][:k]
    return [(chunks[i], combined[i]) for i in top_idxs]


question = "Which technologies were used in the Startup Idea Validation Platform?"
print("Vector-only:")
for c, s in retrieve(question, k=2):
    print(f"  [{s:.3f}] {textwrap.shorten(c.replace(chr(10), ' '), 100)}")

print("\nHybrid (keyword + vector):")
for c, s in hybrid_retrieve(question, k=2):
    print(f"  [{s:.3f}] {textwrap.shorten(c.replace(chr(10), ' '), 100)}")

Vector-only:
  [0.386] Projects Startup Idea Validation Platform | FastAPI, React, Google Gemini API, Docker • [...]
  [0.292] Certifications AWS Academy – Cloud Foundations Alteryx – Data Analytics Process Automation [...]

Hybrid (keyword + vector):
  [0.693] Projects Startup Idea Validation Platform | FastAPI, React, Google Gemini API, Docker • [...]
  [0.278] Technical Skills Programming Languages: Python, Java, C, R, MATLAB (Basic) Web & Backend [...]


## 10. Validation Logs

To evaluate the system, I tested it with questions from different sections of the resume and recorded the retrieved chunks, similarity scores, and generated answers.

In [13]:
test_questions = [
    "What is the educational qualification mentioned in the resume?",
    "What is the CGPA mentioned in the resume?",
    "What projects are described in the resume?",
    "What technologies were used in the Startup Idea Validation Platform?",
    "What programming languages are listed in the resume?",
    "What certifications are listed in the resume?",
    "What achievements are mentioned in the resume?",
    "Which databases are mentioned in the resume?",
]

log_rows = []
for q in test_questions:
    retrieved = hybrid_retrieve(q, k=3)
    top_chunk, top_score = retrieved[0]
    answer = generate_answer(q, [c for c, _ in retrieved])
    log_rows.append({
        "question": q,
        "top_chunk_preview": textwrap.shorten(top_chunk.replace(chr(10), " "), 90),
        "top_similarity": round(top_score, 3),
        "generated_answer": answer,
    })

validation_log = pd.DataFrame(log_rows)
validation_log

,question,top_chunk_preview,top_similarity,generated_answer
0,What is the educational qualification mentioned in the resume?,"Projects Startup Idea Validation Platform | FastAPI, React, Google Gemini API, [...]",0.518,"B.Tech in Computer Science and Engineering (Data Science) CGPA: 8.35/10 Narayana Junior College, Suchitra 2021 – 202..."
1,What is the CGPA mentioned in the resume?,"Education CVR College of Engineering, Hyderabad 2023 – 2027 B.Tech in Computer [...]",0.767,8.35/10
2,What projects are described in the resume?,"Projects Startup Idea Validation Platform | FastAPI, React, Google Gemini API, [...]",0.607,"Startup Idea Validation Platform | FastAPI, React, Google Gemini API, Docker"
3,What technologies were used in the Startup Idea Validation Platform?,"Projects Startup Idea Validation Platform | FastAPI, React, Google Gemini API, [...]",0.698,"FastAPI, React, Google Gemini API, Docker."
4,What programming languages are listed in the resume?,"Technical Skills Programming Languages: Python, Java, C, R, MATLAB (Basic) Web & [...]",0.792,"Python, Java, C, R, MATLAB (Basic), Web & Backend Technologies: Flask, FastAPI, Node.js, React, HTML, CSS, JavaScript"
5,What certifications are listed in the resume?,Certifications AWS Academy – Cloud Foundations Alteryx – Data Analytics Process [...],0.691,AWS Academy – Cloud Foundations Alteryx – Data Analytics Process Automation Virtual Internship AICTE – Altair Data S...
6,What achievements are mentioned in the resume?,Achievements Solved 150+ coding problems on LeetCode focusing on data structures and [...],0.633,Solved 150+ coding problems on LeetCode focusing on data structures and algorithms. CodeChef rating above 1400; acti...
7,Which databases are mentioned in the resume?,"Technical Skills Programming Languages: Python, Java, C, R, MATLAB (Basic) Web & [...]",0.671,"SQL, PL/SQL, PostgreSQL"


In [14]:
validation_log.to_csv("validation_log.csv", index=False)
print("Validation log saved to validation_log.csv")

Validation log saved to validation_log.csv


## 11. System metrics report

In [15]:
system_report = {
    "source_document": SOURCE_FILE,
    "total_characters": len(raw_text),
    "num_chunks": len(chunks),
    "chunking_strategy": "section-heading-based (fallback: line-aware char chunker, 500/100)",
    "avg_chunk_length_chars": round(np.mean([len(c) for c in chunks]), 1),
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimension": EMBED_DIM,
    "vector_store": "FAISS IndexFlatIP (exact cosine via normalized vectors)",
    "retrieval_top_k": 3,
    "hybrid_search": "BM25 (rank_bm25, punctuation-stripped tokenizer) + FAISS cosine, weighted sum (alpha=0.5)",
    "generator_model": "google/flan-t5-base (AutoModelForSeq2SeqLM)",
    "max_context_chars": MAX_CONTEXT_CHARS,
    "num_validation_questions": len(test_questions),
}

for k, v in system_report.items():
    print(f"{k:24s}: {v}")

pd.DataFrame([system_report]).to_csv("system_metrics_report.csv", index=False)

source_document         : Manjunath_SDE_Resume.pdf
total_characters        : 2792
num_chunks              : 7
chunking_strategy       : section-heading-based (fallback: line-aware char chunker, 500/100)
avg_chunk_length_chars  : 398.0
embedding_model         : sentence-transformers/all-MiniLM-L6-v2
embedding_dimension     : 384
vector_store            : FAISS IndexFlatIP (exact cosine via normalized vectors)
retrieval_top_k         : 3
hybrid_search           : BM25 (rank_bm25, punctuation-stripped tokenizer) + FAISS cosine, weighted sum (alpha=0.5)
generator_model         : google/flan-t5-base (AutoModelForSeq2SeqLM)
max_context_chars       : 900
num_validation_questions: 8
